# Step 4b — Web Search: Answering with Live Retrieval

[Closed book](04a_closed_book.ipynb) measured what's in the weights. This
notebook adds the provider's built-in search tool — plus the one prompt
line that travels with it: the model is asked to search before answering
and to report the URLs it relied on in a `citations` field (the
`WebSearchAnswer` schema, a one-field extension of `Answer`). Why the
extra field? Forcing the final message through a structured-output schema
strips OpenAI's usual `url_citation` annotations, so without it the
search condition returns answers with **no receipts**. The two conditions
still differ by exactly one _idea_ — retrieval, with attribution — and
the accuracy gap is the effect of that idea.

As in notebook 4a, we start at the raw SDKs — OpenAI, then Gemini — see
exactly what the search tool changes in each response, then hand it all
back to the toolkit — and finish with this method's most audit-relevant
knob: **domain filtering**.

## 1. The baseline: closed book

Same question, same toolkit call as notebook 4a:


In [1]:
from toolkit import prompts
from toolkit.answers import answer_question
from toolkit.utils import load_jsonl

selected = load_jsonl("../../data/questions/selected_questions.jsonl")
question = selected[0]
user_prompt = prompts.build_answer_user_prompt(
    question["question"], question["options"]
)

closed = answer_question(
    question, model="gpt-5.4-mini-2026-03-17", method="closed_book"
)

print(question["question"], "\n")
for letter, option in zip("ABCD", question["options"]):
    mark = "*" if letter == question["correct_letter"] else " "
    print(f"  {mark}{letter}. {option}")
print(
    f"\nclosed_book: {closed['answer_letter']} "
    f"(confidence {closed['confidence']:.2f}) -> "
    f"{'CORRECT' if closed['is_correct'] else 'WRONG'}"
)

Which company conducted the AI detection review of Pope Leo XIV's collection of speeches and writings, Maps of Hope? 

   A. Breaking News Australia
  *B. Proudly Human
   C. Australian Catholic University
   D. The Vatican Publishing House

closed_book: C (confidence 0.86) -> WRONG


## 2. Turning on search — the raw OpenAI call

At the SDK level, "web search" is one extra argument: `tools`. OpenAI's
Responses API takes a list of tool specs, and `{"type": "web_search"}` is
the entire spec for the built-in search tool. On top of notebook 4a's raw
call we also swap in the web-search variants of the prompt (base prompt +
`ANSWER_WEBSEARCH_ADDENDUM`) and the schema (`WebSearchAnswer`):


In [2]:
from openai import OpenAI

from toolkit.answers import WebSearchAnswer
from toolkit.providers import PROVIDER_ENV, load_api_key

client = OpenAI(api_key=load_api_key(PROVIDER_ENV["openai"]))

websearch_system = prompts.ANSWER_SYSTEM_PROMPT + prompts.ANSWER_WEBSEARCH_ADDENDUM

response = client.responses.parse(
    model="gpt-5.4-mini-2026-03-17",
    input=[
        {"role": "developer", "content": websearch_system},
        {"role": "user", "content": user_prompt},
    ],
    text_format=WebSearchAnswer,  # <- Answer + a `citations` field
    tools=[{"type": "web_search"}],  # <- the search tool itself
)

response.output_parsed

WebSearchAnswer(answer_letter='B', confidence=0.98, reasoning='The review was conducted by Proudly Human, the Australian company that certified *Maps of Hope* as human-authored. The Guardian and ACU both identify Proudly Human as the firm behind the AI detection review.', citations=['https://www.theguardian.com/world/2026/jul/21/pope-leo-speech-human-not-ai-artificial-intelligence', 'https://www.acu.edu.au/about-acu/leadership-and-governance/leadership/vice-chancellor-and-president/maps-of-hope'])

In [3]:
print(response.output_parsed.answer_letter)
print(response.output_parsed.confidence)
print(response.output_parsed.reasoning)
print(response.output_parsed.citations)

B
0.98
The review was conducted by Proudly Human, the Australian company that certified *Maps of Hope* as human-authored. The Guardian and ACU both identify Proudly Human as the firm behind the AI detection review.
['https://www.theguardian.com/world/2026/jul/21/pope-leo-speech-human-not-ai-artificial-intelligence', 'https://www.acu.edu.au/about-acu/leadership-and-governance/leadership/vice-chancellor-and-president/maps-of-hope']


In [4]:
question

{'id': 'gemini__world/2026/jul/21/pope-leo-speech-human-not-ai-artificial-intelligence__q0',
 'article_id': 'world/2026/jul/21/pope-leo-speech-human-not-ai-artificial-intelligence',
 'question_index': 0,
 'provider': 'gemini',
 'model': 'gemini-3.1-flash-lite',
 'question': "Which company conducted the AI detection review of Pope Leo XIV's collection of speeches and writings, Maps of Hope?",
 'options': ['Breaking News Australia',
  'Proudly Human',
  'Australian Catholic University',
  'The Vatican Publishing House'],
 'correct_letter': 'B',
 'explanation': 'The article states that Proudly Human, an Australian company led by Dr. Alan Finkel, undertook the review of the writings and speeches in collaboration with the publisher and other entities.',
 'generated_at': '2026-07-26T16:27:22.020792+00:00',
 'n_models_passing': 3}

Handing a model a tool doesn't force it to use the tool — it can still
answer from its weights. The raw response tells us what actually
happened: its `output` is a list of items, and any item of type
`web_search_call` is a search the model chose to run:


In [3]:
raw = response.model_dump(mode="json", warnings=False)
for item in raw["output"]:
    line = f"  {item['type']}"
    if item["type"] == "web_search_call":
        query = (item.get("action") or {}).get("query")
        line += f"  query={query!r}"
    print(line)

search_used = any(i["type"] == "web_search_call" for i in raw["output"])
print("\nsearch_used =", search_used)

  web_search_call  query='Pope Leo XIV Maps of Hope AI detection review company conducted review speeches writings Maps of Hope'
  message

search_used = True


## 3. The same call on Gemini

Gemini's dialect of the same idea: the built-in tool is called
`google_search`, and it goes inside the `GenerateContentConfig` next to
the schema — the same two deltas (tool + citation contract) as the OpenAI
call. The search evidence lives in a different place too: not output
items, but `grounding_metadata` on the response candidate, which records
the queries the model ran _and_ the sources that grounded the answer —
so on Gemini the self-reported `citations` sit alongside real,
API-verified attribution:


In [5]:
from google import genai
from google.genai import types

gclient = genai.Client(api_key=load_api_key(PROVIDER_ENV["gemini"]))

gresponse = gclient.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=user_prompt,
    config=types.GenerateContentConfig(
        system_instruction=websearch_system,
        response_mime_type="application/json",
        response_schema=WebSearchAnswer,
        tools=[types.Tool(google_search=types.GoogleSearch())],  # <- the one new line
    ),
)

ganswer = gresponse.parsed
grounding = gresponse.candidates[0].grounding_metadata
queries = grounding.web_search_queries if grounding else None

print(f"Gemini: {ganswer.answer_letter} ({ganswer.confidence:.2f})")
print("search queries:", queries)
print("search_used =", bool(queries))
print("self-reported citations:", ganswer.citations)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Gemini: B (1.00)
search queries: None
search_used = False
self-reported citations: ['https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEv5_pD2f5e5KnNdfneMohdca_AgbOZY-KAZg6RxOgOZUXXwg9kFuhVJRY8B0h0QJwXZfqd_W0AgVXYVUWr3uem8drKO4e2OlaqEBaAsn4vq_GoCYQf80y9fxLIfS_RJlZ7q_dLrB0GaF3sIKpWL8jb0X5QVeRnjZmAYhU4lv4iymypGyKBP8P2rRMz6lopXchtSKNodtELyQ==', 'https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFBaEcmAdypy9vVkCqvdRP_EP2T-Cov2EyXqoSzXc7yYoAYEbfJhVci5uxcfRo62aErDzCGncS6VT-zjTxHpjjqzfUHbQuZS3t_BZKZc88i4aqoFegXHGafJNP0IjFaWQZCBZc8sy7JLnmixvttOt8tC2m2TBeCGrbtYd4wd72zoDbwnCN9ppg6vxDxzXoW6DT5wkdOTyksPsrQPBlX']


In [6]:
from toolkit.utils import resolve_redirect_urls

real_urls = resolve_redirect_urls(ganswer.citations)

In [9]:
for orig, real in real_urls.items():
    print(f"{orig}")
    print(f"  -> {real}")

https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEv5_pD2f5e5KnNdfneMohdca_AgbOZY-KAZg6RxOgOZUXXwg9kFuhVJRY8B0h0QJwXZfqd_W0AgVXYVUWr3uem8drKO4e2OlaqEBaAsn4vq_GoCYQf80y9fxLIfS_RJlZ7q_dLrB0GaF3sIKpWL8jb0X5QVeRnjZmAYhU4lv4iymypGyKBP8P2rRMz6lopXchtSKNodtELyQ==
  -> https://www.theguardian.com/world/2026/jul/21/pope-leo-speech-human-not-ai-artificial-intelligence
https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFBaEcmAdypy9vVkCqvdRP_EP2T-Cov2EyXqoSzXc7yYoAYEbfJhVci5uxcfRo62aErDzCGncS6VT-zjTxHpjjqzfUHbQuZS3t_BZKZc88i4aqoFegXHGafJNP0IjFaWQZCBZc8sy7JLnmixvttOt8tC2m2TBeCGrbtYd4wd72zoDbwnCN9ppg6vxDxzXoW6DT5wkdOTyksPsrQPBlX
  -> https://www.proudlyhuman.org/news-room/media-releases/pope-leo-xiv-s-writings-verified-by-proudly-human


## 4. This is all in the toolkit

`answer_question(method="web_search")` runs the same call — the provider
adapters take a `use_web_search` flag that adds the tool spec, while
`toolkit.answers` appends the citation addendum and swaps in
`WebSearchAnswer` — detects search use, and grades the answer into the
standard record, which now carries the model's self-reported `citations`:


In [10]:
web = answer_question(question, model="gpt-5.4-mini-2026-03-17", method="web_search")

print(
    f"closed_book : {closed['answer_letter']} (confidence {closed['confidence']:.2f})"
)
print(
    f"web_search  : {web['answer_letter']} "
    f"(confidence {web['confidence']:.2f}, "
    f"searched: {web['search_used']})"
)
print("WHY:", web["reasoning"])
print("CITED:", web["citations"])

closed_book : C (confidence 0.86)
web_search  : B (confidence 0.98, searched: True)
WHY: The review of *Maps of Hope* was conducted by the Australian company Proudly Human, which the reporting describes as certifying the collection as human-authored. That matches option B, not the other organizations listed.
CITED: ['https://www.theguardian.com/world/2026/jul/21/pope-leo-speech-human-not-ai-artificial-intelligence', 'https://en.prnasia.com/releases/apac/pope-leo-xiv-s-writings-verified-by-proudly-human-541397.shtml']


## 5. Parameter deep-dive: choosing where the model can look

"Web search" sounds like one condition, but _which web_ matters. OpenAI's
`web_search` tool takes a `filters` object with two lists (up to 100
domains each, written without the `https://` scheme):

```python
tools=[{
    "type": "web_search",
    "filters": {
        "allowed_domains": ["theguardian.com"],   # may ONLY cite these
        # "blocked_domains": ["theguardian.com"], # may cite anything BUT these
    },
}]
```

The toolkit exposes them as `include_domains` and `exclude_domains` on
`answer_question()`. That turns retrieval into a controlled variable, and
our quiz has a perfect stress test built in: **every question was written
from a Guardian article.**

- `include_domains=["theguardian.com"]` points the model straight at the
  source outlet — an open-book exam where we chose the book.
- `exclude_domains=["theguardian.com"]` takes the source away — can the
  model corroborate the fact anywhere _else_ on the web?

(One asymmetry to know: this is an OpenAI-only knob. The Gemini
Developer API has no domain filters — `GoogleSearch.exclude_domains`
exists in the SDK but is Enterprise-only — so the toolkit raises a
`ValueError` if you try, rather than silently ignoring the filter.)


In [11]:
included = answer_question(
    question,
    model="gpt-5.4-mini-2026-03-17",
    method="web_search",
    include_domains=["theguardian.com"],
)

print(
    f"guardian only: {included['answer_letter']} "
    f"(confidence {included['confidence']:.2f}, "
    f"searched: {included['search_used']})"
)
print("WHY:", included["reasoning"])

guardian only: B (confidence 0.98, searched: True)
WHY: The Guardian says the review was undertaken by the Australian company Proudly Human. That matches option B exactly.


In [12]:
excluded = answer_question(
    question,
    model="gpt-5.4-mini-2026-03-17",
    method="web_search",
    exclude_domains=["theguardian.com"],
)

runs = [
    ("closed_book", closed),
    ("web (unrestricted)", web),
    ("web (guardian only)", included),
    ("web (guardian blocked)", excluded),
]
print(f"correct letter: {question['correct_letter']}\n")
for label, r in runs:
    verdict = "CORRECT" if r["is_correct"] else "WRONG"
    searched = r.get("search_used", "-")
    print(
        f"{label:<24} {r['answer_letter']} "
        f"(confidence {r['confidence']:.2f}, searched: {searched}) "
        f"-> {verdict}"
    )

correct letter: B

closed_book              C (confidence 0.86, searched: None) -> WRONG
web (unrestricted)       B (confidence 0.98, searched: True) -> CORRECT
web (guardian only)      B (confidence 0.98, searched: True) -> CORRECT
web (guardian blocked)   B (confidence 0.98, searched: True) -> CORRECT


## 6. The full experiment, from the command line

The sweep runs unfiltered — the open web is the condition being measured;
the filters above are the notebook's microscope, not part of the
pipeline:

```bash
# 04-1: web search
for M in gpt-5.4-mini-2026-03-17 gemini-3.1-flash-lite; do
  uv run python scripts/04-1_generate_answers.py --model $M --method web_search --parallel
done
```


## 7. The map

| This notebook              | Where it lives                                                                                                                                      |
| -------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------- |
| §2 the OpenAI tool spec    | `toolkit.providers._keys.OPENAI_WEBSEARCH_TOOLS`; the `use_web_search` flag on `openai_provider.run_parsed()`                                       |
| §2–3 the citation contract | `toolkit.prompts.ANSWER_WEBSEARCH_ADDENDUM`; `toolkit.answers.WebSearchAnswer`                                                                      |
| §3 the Gemini tool spec    | `toolkit.providers.gemini_provider.GEMINI_WEBSEARCH_TOOLS`; same flag on `gemini_provider.run_parsed()`                                             |
| §2–3 search detection      | `toolkit.answers._detect_search_use()`                                                                                                              |
| §4 one graded record       | `toolkit.answers.answer_question()`                                                                                                                 |
| §5 domain filters          | the `include_domains` / `exclude_domains` kwargs on `answer_question()`; tool built in `toolkit.providers.openai_provider._build_websearch_tools()` |
| §6 at scale                | `scripts/04-1_generate_answers.py`; `toolkit.answers.answer_questions()`                                                                            |

---

### Next up 🗣️

The last method gives the model no tools at all — just company:
[`04c_debate.ipynb`](04c_debate.ipynb) makes three copies of the model
argue it out.
